Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Load the model

In [3]:
from google import genai
from google.genai import types
import base64

client = userdata.get('GOOGLE_API_KEY')

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=["Say hello."]
)
print(response.text)

Hello!


Install independencies

In [4]:
!pip install google-genai --break-system-packages

Hm full step1 v2 v3

In [4]:
"""
HM FULL RUN — Step 1 (both V2 and V3) over all 239 test images
==============================================================
Produces the two Step-1 evidence files that Step 2 will consume to fill the
"Scene Graph without CoT" (V2) and "Scene Graph with CoT" (V3) rows of the
HM results table.

Key properties
--------------
- Reads image ids + GT from Test_labels.xlsx (image_id forced to int so the
  path builder never yields '1.0.jpg').
- Runs V2 and V3 for every image, writing to two separate files:
    step1_v2_full.json   (flat HSL, no reasoning fields)
    step1_v3_full.json   (nested HSL with per-step reasoning)
- V2 prompt is PATCHED: the clinical/formal-terms clarification that was
  previously only in V3 is now added to V2's coded_language section, so the
  two conditions share the same field definition and the V2-vs-V3 comparison
  is clean (this was the known field-definition inconsistency).
- Token config: V3 uses 8192 / thinking_budget=2048 for ALL images (so dense
  Traditional-Chinese memes don't truncate and get silently excluded). V2 is
  the no-reasoning condition: thinking_budget=0, 4096 output.
- Checkpoint after every call, per version. Safe to re-run after a Colab
  timeout — completed (image_id, version) pairs are skipped.
- Failures are recorded with _step1_success=false and excluded downstream
  per the ERROR-exclusion policy; a summary lists them for optional retry.

Run in Google Colab with Google Drive mounted.
"""

import json
import os
import time

import pandas as pd
from google import genai
from google.genai import types

# ============================================================
# CONFIG
# ============================================================


BASE_DIR = "/content/drive/MyDrive/MyThesis2026/Chinese/Test"
IMAGE_DIR = os.path.join(BASE_DIR, "Test_images")
LABELS_PATH = os.path.join(BASE_DIR, "Test_labels.xlsx")

MODEL_NAME = "gemini-2.5-flash"
MAX_RETRIES = 3
RETRY_DELAY = 15

V2_OUT = os.path.join(BASE_DIR, "step1_v2_full.json")
V3_OUT = os.path.join(BASE_DIR, "step1_v3_full.json")

# Per-version generation settings
V2_MAX_TOKENS = 4096
V3_MAX_TOKENS = 8192
V3_THINKING = 2048

# ============================================================
# V2 PROMPT — Scene Graph WITHOUT CoT (PATCHED)
# The only change vs the original V2 is the inserted clinical-terms
# clarification (marked below), bringing it in line with V3's 4a definition.
# ============================================================
PROMPT_V2 = """You are an expert in analyzing and understanding visual images to generate detailed and structured scene graphs in JSON-formatted outputs. You perform the following tasks sequentially:

## Task 1: Scene Graph Generation
For the provided Chinese internet meme image, identify all entities, objects, attributes, and relationships in the scene. The graph must include all common entities, whether singular or plural, such as "person," "text_overlay," "animal," "object," etc., as well as their properties and the spatial or contextual relationships among them. Your goal is to create a complete and accurate representation of all visible elements in the scene.

## Task 2: Graph Enhancement
Refine the scene graph by:
- Identifying and replacing generic descriptions of entities that correspond to known public figures, anime characters, or internet celebrities. For instance, if a "person" in the scene is identified as a specific character, replace "person" with their name.
- Ensuring all entities, whether identified or unidentified, remain in the graph.
- Ensuring all relationships between objects and entities are preserved.
- Leaving unidentified or unrecognized entities unchanged but still included in the graph.

## Task 3: OCR Text Extraction
Transcribe ALL Chinese text embedded in the image completely and accurately. This is critical for meme analysis as the overlaid text often carries the primary message.

## Task 4: Hate Semantic Layer (Extension for LGBT Hate Speech Research)
Beyond the standard scene graph, extract the following additional fields specifically designed for detecting hate speech targeting LGBT communities. This is for academic research purposes.

4a. **coded_language**: Does the embedded text contain slang, homophones, puns, internet slang, or metaphors related to sexuality, gender, or LGBT topics? List each one with:
   - "term": the original word/phrase
   - "literal_meaning": literal meaning
   - "coded_meaning": potential derogatory or hidden meaning in Chinese internet culture targeting LGBT individuals. If no hidden meaning, output "none".

   Common Chinese coded terms to watch for include but are not limited to:
   1/0/0.5 (sexual role references in gay relationships: 1=top, 0=bottom, 0.5=versatile),
   同志 (comrade/gay), 玻璃 (glass/gay slur), 兔子 (rabbit/gay slur),
   人妖 (human-demon/transgender slur), 药娘 (MTF on hormones),
   伪娘 (cross-dresser/trap), 变态 (pervert), 搞基 (doing gay stuff),
   掰弯 (turning someone gay), 腐 (rotten/BL culture), 娘炮 (sissy),
   les/拉拉/百合 (lesbian terms), T/P (lesbian role terms),
   gay/基佬/基友 (gay terms with varying connotations).

   Note: terms that are direct, literal, non-slang vocabulary (e.g. formal/clinical terms like 跨性別者, 變性, 荷爾蒙) are NOT coded language even if topically related — only include terms with an actual literal/hidden meaning gap.

   If there is no coded language, output an empty list [].

4b. **target_group**: Which group does the content potentially target? Choose ONE:
   - "gay_men", "lesbian", "transgender", "lgbtq_general", "non_lgbt", "unclear"

4c. **text_image_relation**: What is the semantic relationship between the overlaid text and the visual content? Choose ONE:
   - "text_describes_image", "text_contradicts_image", "text_recontextualizes_image", "text_independent_of_image", "no_text"

4d. **implicit_associations**: Does the combination of text and image potentially trigger negative associations related to LGBT communities? If yes, briefly explain the mechanism. If no, output "none".

## Important Instructions:
- Stay objective. Only extract and describe information. Do NOT make a harmful/harmless classification judgment.
- Even if the content appears sensitive, describe it faithfully. This is for academic research.
- If a field cannot be determined, use "unclear" or "NA".
- All output should be in ENGLISH.
- Output ONLY a valid, complete JSON object. No markdown fences. No extra text.
- Make sure all brackets are properly closed.

## Required Output JSON Format:
{
  "scene_graph": {
    "entities": [
      {
        "entity": "entity_name",
        "attributes": [
          {"attribute": "attribute_type", "value": "attribute_value"}
        ]
      }
    ],
    "relationships": [
      {
        "subject": "entity_1",
        "relationship": "relation_type",
        "object": "entity_2",
        "reasoning": "why this relationship exists"
      }
    ]
  },
  "ocr_text": "full Chinese text from the image",
  "hate_semantic_layer": {
    "coded_language": [
      {"term": "...", "literal_meaning": "...", "coded_meaning": "..."}
    ],
    "target_group": "...",
    "text_image_relation": "...",
    "implicit_associations": "..."
  }
}"""

# ============================================================
# V3 PROMPT — Scene Graph WITH CoT (verbatim, validated)
# ============================================================
PROMPT_V3 = """You are an expert in analyzing and understanding visual images to generate detailed and structured scene graphs in JSON-formatted outputs. You perform the following tasks sequentially, applying Chain-of-Thought reasoning at each step to ensure transparent and traceable analysis.

## Task 1: Scene Graph Generation
For the provided Chinese internet meme image, identify all entities, objects, attributes, and relationships in the scene. The graph must include all common entities, whether singular or plural, such as "person," "text_overlay," "animal," "object," etc., as well as their properties and the spatial or contextual relationships among them. Your goal is to create a complete and accurate representation of all visible elements in the scene.

## Task 2: Graph Enhancement
Refine the scene graph by:
- Identifying and replacing generic descriptions of entities that correspond to known public figures, anime characters, or internet celebrities. For instance, if a "person" in the scene is identified as a specific character, replace "person" with their name.
- Ensuring all entities, whether identified or unidentified, remain in the graph.
- Ensuring all relationships between objects and entities are preserved.
- Leaving unidentified or unrecognized entities unchanged but still included in the graph.

## Task 3: OCR Text Extraction
Transcribe ALL Chinese text embedded in the image completely and accurately. This is critical for meme analysis as the overlaid text often carries the primary message.

## Task 4: Hate Semantic Layer with Chain-of-Thought Reasoning
Beyond the standard scene graph, perform a step-by-step Chain-of-Thought analysis specifically designed for detecting hate speech targeting LGBT communities. This is for academic research purposes.

You MUST reason through the following steps sequentially. For each step, first explain your reasoning in the "reasoning" field, then provide your conclusion in the corresponding output field. Keep each reasoning field concise (2-4 sentences) so the full output fits within the response budget.

**Step 4a — Coded Language Analysis**
Think carefully: Does the embedded text contain slang, homophones, puns, internet slang, or metaphors related to sexuality, gender, or LGBT topics?

For each potential term, briefly reason through: (i) literal surface meaning, (ii) secondary meaning in Chinese internet culture targeting LGBT individuals, (iii) why it does or does not carry a coded meaning.

Common Chinese coded terms to watch for include but are not limited to:
1/0/0.5 (sexual role references in gay relationships: 1=top, 0=bottom, 0.5=versatile),
同志 (comrade/gay), 玻璃 (glass/gay slur), 兔子 (rabbit/gay slur),
人妖 (human-demon/transgender slur), 药娘 (MTF on hormones),
伪娘 (cross-dresser/trap), 变态 (pervert), 搞基 (doing gay stuff),
掰弯 (turning someone gay), 腐 (rotten/BL culture), 娘炮 (sissy),
les/拉拉/百合 (lesbian terms), T/P (lesbian role terms),
gay/基佬/基友 (gay terms with varying connotations).

Note: terms that are direct, literal, non-slang vocabulary (e.g. formal/clinical terms like 跨性別者, 變性, 荷爾蒙) are NOT coded language even if topically related — only include terms with an actual literal/hidden meaning gap.

**Step 4b — Target Group Identification**
Choose ONE: "gay_men", "lesbian", "transgender", "lgbtq_general", "non_lgbt", "unclear". Briefly justify using visual/textual cues and any coded language from 4a.

**Step 4c — Text-Image Relationship Analysis**
Choose ONE: "text_describes_image", "text_contradicts_image", "text_recontextualizes_image", "text_independent_of_image", "no_text". Briefly justify.

**Step 4d — Implicit Association Analysis**
Considering all previous steps together, does the combination potentially trigger negative associations related to LGBT communities? Briefly explain the mechanism, or state "none".

## Important Instructions:
- Stay objective. Only extract, describe, and reason about information. Do NOT make a final harmful/harmless classification judgment — that will be done in a separate step.
- Even if the content appears sensitive, describe and reason about it faithfully. This is for academic research.
- If a field cannot be determined, use "unclear" or "NA".
- All output should be in ENGLISH.
- Keep reasoning fields concise — this is critical to avoid truncation.
- Output ONLY a valid, complete JSON object. No markdown fences. No extra text.
- Make sure all brackets are properly closed.

## Required Output JSON Format:
{
  "scene_graph": {
    "entities": [
      {"entity": "entity_name", "attributes": [{"attribute": "attribute_type", "value": "attribute_value"}]}
    ],
    "relationships": [
      {"subject": "entity_1", "relationship": "relation_type", "object": "entity_2", "reasoning": "why this relationship exists"}
    ]
  },
  "ocr_text": "full Chinese text from the image",
  "hate_semantic_layer": {
    "step_4a_coded_language": {
      "reasoning": "concise step-by-step analysis",
      "coded_terms": [{"term": "...", "literal_meaning": "...", "coded_meaning": "..."}]
    },
    "step_4b_target_group": {
      "reasoning": "concise justification",
      "target": "one of: gay_men / lesbian / transgender / lgbtq_general / non_lgbt / unclear"
    },
    "step_4c_text_image_relation": {
      "reasoning": "concise justification",
      "relation": "one of: text_describes_image / text_contradicts_image / text_recontextualizes_image / text_independent_of_image / no_text"
    },
    "step_4d_implicit_associations": {
      "reasoning": "concise synthesis",
      "association": "description of the negative association mechanism, or none"
    }
  }
}"""

# Per-version dispatch table
VERSIONS = {
    "v2": {"prompt": PROMPT_V2, "out": V2_OUT,
           "max_tokens": V2_MAX_TOKENS, "thinking": 0},
    "v3": {"prompt": PROMPT_V3, "out": V3_OUT,
           "max_tokens": V3_MAX_TOKENS, "thinking": V3_THINKING},
}


# ============================================================
# HELPERS
# ============================================================
def load_labels(path):
    df = pd.read_excel(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    if "image_id" not in df.columns or "class" not in df.columns:
        raise ValueError(f"Expected columns image_id,class — got {list(df.columns)}")
    out = []
    for _, row in df.iterrows():
        try:
            iid = int(row["image_id"])
        except (ValueError, TypeError):
            continue
        out.append({"image_id": iid, "gt": str(row["class"]).strip()})
    out.sort(key=lambda x: x["image_id"])
    return out


def find_image(img_id):
    for ext in [".jpg", ".jpeg", ".png", ".webp"]:
        p = os.path.join(IMAGE_DIR, f"{img_id}{ext}")
        if os.path.exists(p):
            return p
    return None


def call_step1(client, image_path, prompt, max_tokens, thinking):
    with open(image_path, "rb") as f:
        image_data = f.read()
    ext = os.path.splitext(image_path)[1].lower()
    mime_type = "image/png" if ext == ".png" else "image/jpeg"

    thinking_cfg = types.ThinkingConfig(thinking_budget=thinking)

    raw = ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=[
                    types.Part.from_bytes(data=image_data, mime_type=mime_type),
                    prompt,
                ],
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    max_output_tokens=max_tokens,
                    thinking_config=thinking_cfg,
                ),
            )
            raw = response.text.strip()
            cleaned = raw
            if cleaned.startswith("```json"):
                cleaned = cleaned[7:]
            elif cleaned.startswith("```"):
                cleaned = cleaned[3:]
            if cleaned.endswith("```"):
                cleaned = cleaned[:-3]
            cleaned = cleaned.strip()
            return json.loads(cleaned), True, raw
        except json.JSONDecodeError:
            if attempt < MAX_RETRIES:
                print(f"      JSON parse failed ({attempt}/{MAX_RETRIES}), "
                      f"len={len(raw)}, tail=...{raw[-100:]}")
                time.sleep(RETRY_DELAY)
            else:
                return None, False, raw
        except Exception as e:
            msg = str(e)
            if "503" in msg or "UNAVAILABLE" in msg or "429" in msg:
                wait = RETRY_DELAY * attempt
                print(f"      Server error ({attempt}), waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"      NON-RETRYABLE: {msg[:200]}")
                return None, False, msg
    return None, False, "Max retries exceeded"


def load_ckpt(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        done = {r["image_id"] for r in data if r.get("_step1_success")}
        return data, done
    return [], set()


def save(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


# ============================================================
# MAIN
# ============================================================
def main():
    client = genai.Client(api_key=API_KEY)
    samples = load_labels(LABELS_PATH)
    print(f"Loaded {len(samples)} images from {os.path.basename(LABELS_PATH)}.\n")

    for ver, cfg in VERSIONS.items():
        print(f"\n{'#'*70}\n# STEP 1 — {ver.upper()}  (out: {os.path.basename(cfg['out'])})\n{'#'*70}")
        results, done = load_ckpt(cfg["out"])
        if done:
            print(f"Resuming {ver}: {len(done)} already done, skipping.\n")

        failed = []
        for s in samples:
            img_id, gt = s["image_id"], s["gt"]
            if img_id in done:
                continue
            image_path = find_image(img_id)
            if not image_path:
                print(f"[SKIP] {ver} image {img_id}: file not found")
                results.append({"image_id": img_id, "ground_truth": gt,
                                "_step1_success": False, "_error": "file not found"})
                failed.append(img_id)
                save(cfg["out"], results)
                continue

            parsed, ok, raw = call_step1(
                client, image_path, cfg["prompt"], cfg["max_tokens"], cfg["thinking"])

            if ok:
                parsed["image_id"] = img_id
                parsed["ground_truth"] = gt
                parsed["_step1_success"] = True
                results.append(parsed)
                print(f"  {ver} img {img_id:>4} OK")
            else:
                results.append({"image_id": img_id, "ground_truth": gt,
                                "_step1_success": False, "_error": raw[:300]})
                failed.append(img_id)
                print(f"  {ver} img {img_id:>4} FAILED — {raw[:100]}")

            save(cfg["out"], results)
            time.sleep(4)

        ok_n = sum(1 for r in results if r.get("_step1_success"))
        print(f"\n{ver.upper()} done: {ok_n}/{len(samples)} succeeded.")
        if failed:
            print(f"  {ver} failures (excluded, re-runnable): {sorted(set(failed))}")

    print(f"\nStep 1 full run complete.\n  V2 -> {V2_OUT}\n  V3 -> {V3_OUT}")
    print("Next: run the Step 2 full script to produce both HM table rows.")


if __name__ == "__main__":
    main()

Loaded 239 images from Test_labels.xlsx.


######################################################################
# STEP 1 — V2  (out: step1_v2_full.json)
######################################################################
  v2 img    1 OK
  v2 img    2 OK
  v2 img    3 OK
  v2 img    4 OK
  v2 img    5 OK
  v2 img    6 OK
  v2 img    7 OK
  v2 img    8 OK
  v2 img    9 OK
  v2 img   10 OK
  v2 img   11 OK
  v2 img   12 OK
[SKIP] v2 image 13: file not found
[SKIP] v2 image 14: file not found
  v2 img   15 OK
  v2 img   16 OK
      Server error (1), waiting 15s...
  v2 img   17 OK
  v2 img   18 OK
  v2 img   19 OK
      Server error (1), waiting 15s...
  v2 img   20 OK
  v2 img   21 OK
  v2 img   22 OK
  v2 img   23 OK
  v2 img   24 OK
  v2 img   25 OK
  v2 img   26 OK
  v2 img   27 OK
  v2 img   28 OK
  v2 img   29 OK
  v2 img   30 OK
  v2 img   31 OK
  v2 img   32 OK
  v2 img   33 OK
  v2 img   34 OK
  v2 img   35 OK
  v2 img   36 OK
  v2 img   37 OK
  v2 img   38 OK
  v2 img   39 

补跑查看格式

In [5]:
import os
IMAGE_DIR = "/content/drive/MyDrive/MyThesis2026/Chinese/Test/Test_images"
for m in [13, 14, 129, 137, 179, 199, 221, 230]:
    hits = [f for f in os.listdir(IMAGE_DIR) if f.startswith(str(m))]
    print(m, "->", hits)

13 -> ['139.jpg', '134.jpg', '13.gif', '132.jpg', '138.jpg', '133.jpg', '135.webp', '131.jpg', '137.gif', '130.jpg', '136.jpg']
14 -> ['143.jpg', '14.gif', '146.jpg', '148.jpg', '145.webp', '142.jpeg', '149.jpeg', '147.jpg', '144.jpeg', '140.jpg', '141.jpg']
129 -> ['129.gif']
137 -> ['137.gif']
179 -> ['179.gif']
199 -> ['199.gif']
221 -> ['221.gif']
230 -> ['230.gif']


补跑刚刚没有跑的

In [6]:
"""
HM Step 1 BACKFILL — only the missing/failed images
===================================================
Does NOT rerun the whole set. Reruns only:
  - 8 GIFs skipped because the extension list lacked .gif:
      13, 14, 129, 137, 179, 199, 221, 230   (both V2 and V3)
  - V2 truncation failure: 224
  - V3 failures: 114 (503 exhausted), 175 (truncation at ~22k chars)

Fixes applied here:
  * find_image now includes .gif (and upper-case variants).
  * GIFs are loaded, seeked to frame 0, and converted to PNG in-memory
    before being sent — so the multi-frame/format issues never arise and
    the request path matches every other (static) image.
  * Raised output-token ceilings for the backfill, since two of the
    failures were genuine truncations on very dense memes:
      V2 -> 6144, V3 -> 16384 (175 alone needed >8192).

Results are MERGED into the existing files:
    step1_v2_full.json , step1_v3_full.json
Any prior failed/placeholder entry for a backfilled id is removed first,
then the fresh successful record is inserted and the list re-sorted.

Run in Google Colab with Google Drive mounted.
"""

import io
import json
import os
import time

from PIL import Image
from google import genai
from google.genai import types

# ============================================================
# CONFIG
# ============================================================


BASE_DIR = "/content/drive/MyDrive/MyThesis2026/Chinese/Test"
IMAGE_DIR = os.path.join(BASE_DIR, "Test_images")
LABELS_PATH = os.path.join(BASE_DIR, "Test_labels.xlsx")

MODEL_NAME = "gemini-2.5-flash"
MAX_RETRIES = 4          # a touch higher for the stubborn 503 case (114)
RETRY_DELAY = 15

V2_OUT = os.path.join(BASE_DIR, "step1_v2_full.json")
V3_OUT = os.path.join(BASE_DIR, "step1_v3_full.json")

# Raised ceilings for the backfill specifically
V2_MAX_TOKENS = 6144
V3_MAX_TOKENS = 16384
V3_THINKING = 2048

# Which ids to backfill per version
GIFS = [13, 14, 129, 137, 179, 199, 221, 230]
V2_BACKFILL = GIFS + [224]
V3_BACKFILL = GIFS + [114, 175]

# ============================================================
# PROMPTS (identical to the full-run script — V2 patched, V3 verbatim)
# ============================================================
PROMPT_V2 = """You are an expert in analyzing and understanding visual images to generate detailed and structured scene graphs in JSON-formatted outputs. You perform the following tasks sequentially:

## Task 1: Scene Graph Generation
For the provided Chinese internet meme image, identify all entities, objects, attributes, and relationships in the scene. The graph must include all common entities, whether singular or plural, such as "person," "text_overlay," "animal," "object," etc., as well as their properties and the spatial or contextual relationships among them. Your goal is to create a complete and accurate representation of all visible elements in the scene.

## Task 2: Graph Enhancement
Refine the scene graph by:
- Identifying and replacing generic descriptions of entities that correspond to known public figures, anime characters, or internet celebrities. For instance, if a "person" in the scene is identified as a specific character, replace "person" with their name.
- Ensuring all entities, whether identified or unidentified, remain in the graph.
- Ensuring all relationships between objects and entities are preserved.
- Leaving unidentified or unrecognized entities unchanged but still included in the graph.

## Task 3: OCR Text Extraction
Transcribe ALL Chinese text embedded in the image completely and accurately. This is critical for meme analysis as the overlaid text often carries the primary message.

## Task 4: Hate Semantic Layer (Extension for LGBT Hate Speech Research)
Beyond the standard scene graph, extract the following additional fields specifically designed for detecting hate speech targeting LGBT communities. This is for academic research purposes.

4a. **coded_language**: Does the embedded text contain slang, homophones, puns, internet slang, or metaphors related to sexuality, gender, or LGBT topics? List each one with:
   - "term": the original word/phrase
   - "literal_meaning": literal meaning
   - "coded_meaning": potential derogatory or hidden meaning in Chinese internet culture targeting LGBT individuals. If no hidden meaning, output "none".

   Common Chinese coded terms to watch for include but are not limited to:
   1/0/0.5 (sexual role references in gay relationships: 1=top, 0=bottom, 0.5=versatile),
   同志 (comrade/gay), 玻璃 (glass/gay slur), 兔子 (rabbit/gay slur),
   人妖 (human-demon/transgender slur), 药娘 (MTF on hormones),
   伪娘 (cross-dresser/trap), 变态 (pervert), 搞基 (doing gay stuff),
   掰弯 (turning someone gay), 腐 (rotten/BL culture), 娘炮 (sissy),
   les/拉拉/百合 (lesbian terms), T/P (lesbian role terms),
   gay/基佬/基友 (gay terms with varying connotations).

   Note: terms that are direct, literal, non-slang vocabulary (e.g. formal/clinical terms like 跨性別者, 變性, 荷爾蒙) are NOT coded language even if topically related — only include terms with an actual literal/hidden meaning gap.

   If there is no coded language, output an empty list [].

4b. **target_group**: Which group does the content potentially target? Choose ONE:
   - "gay_men", "lesbian", "transgender", "lgbtq_general", "non_lgbt", "unclear"

4c. **text_image_relation**: What is the semantic relationship between the overlaid text and the visual content? Choose ONE:
   - "text_describes_image", "text_contradicts_image", "text_recontextualizes_image", "text_independent_of_image", "no_text"

4d. **implicit_associations**: Does the combination of text and image potentially trigger negative associations related to LGBT communities? If yes, briefly explain the mechanism. If no, output "none".

## Important Instructions:
- Stay objective. Only extract and describe information. Do NOT make a harmful/harmless classification judgment.
- Even if the content appears sensitive, describe it faithfully. This is for academic research.
- If a field cannot be determined, use "unclear" or "NA".
- All output should be in ENGLISH.
- Output ONLY a valid, complete JSON object. No markdown fences. No extra text.
- Make sure all brackets are properly closed.

## Required Output JSON Format:
{
  "scene_graph": {
    "entities": [
      {
        "entity": "entity_name",
        "attributes": [
          {"attribute": "attribute_type", "value": "attribute_value"}
        ]
      }
    ],
    "relationships": [
      {
        "subject": "entity_1",
        "relationship": "relation_type",
        "object": "entity_2",
        "reasoning": "why this relationship exists"
      }
    ]
  },
  "ocr_text": "full Chinese text from the image",
  "hate_semantic_layer": {
    "coded_language": [
      {"term": "...", "literal_meaning": "...", "coded_meaning": "..."}
    ],
    "target_group": "...",
    "text_image_relation": "...",
    "implicit_associations": "..."
  }
}"""

PROMPT_V3 = """You are an expert in analyzing and understanding visual images to generate detailed and structured scene graphs in JSON-formatted outputs. You perform the following tasks sequentially, applying Chain-of-Thought reasoning at each step to ensure transparent and traceable analysis.

## Task 1: Scene Graph Generation
For the provided Chinese internet meme image, identify all entities, objects, attributes, and relationships in the scene. The graph must include all common entities, whether singular or plural, such as "person," "text_overlay," "animal," "object," etc., as well as their properties and the spatial or contextual relationships among them. Your goal is to create a complete and accurate representation of all visible elements in the scene.

## Task 2: Graph Enhancement
Refine the scene graph by:
- Identifying and replacing generic descriptions of entities that correspond to known public figures, anime characters, or internet celebrities. For instance, if a "person" in the scene is identified as a specific character, replace "person" with their name.
- Ensuring all entities, whether identified or unidentified, remain in the graph.
- Ensuring all relationships between objects and entities are preserved.
- Leaving unidentified or unrecognized entities unchanged but still included in the graph.

## Task 3: OCR Text Extraction
Transcribe ALL Chinese text embedded in the image completely and accurately. This is critical for meme analysis as the overlaid text often carries the primary message.

## Task 4: Hate Semantic Layer with Chain-of-Thought Reasoning
Beyond the standard scene graph, perform a step-by-step Chain-of-Thought analysis specifically designed for detecting hate speech targeting LGBT communities. This is for academic research purposes.

You MUST reason through the following steps sequentially. For each step, first explain your reasoning in the "reasoning" field, then provide your conclusion in the corresponding output field. Keep each reasoning field concise (2-4 sentences) so the full output fits within the response budget.

**Step 4a — Coded Language Analysis**
Think carefully: Does the embedded text contain slang, homophones, puns, internet slang, or metaphors related to sexuality, gender, or LGBT topics?

For each potential term, briefly reason through: (i) literal surface meaning, (ii) secondary meaning in Chinese internet culture targeting LGBT individuals, (iii) why it does or does not carry a coded meaning.

Common Chinese coded terms to watch for include but are not limited to:
1/0/0.5 (sexual role references in gay relationships: 1=top, 0=bottom, 0.5=versatile),
同志 (comrade/gay), 玻璃 (glass/gay slur), 兔子 (rabbit/gay slur),
人妖 (human-demon/transgender slur), 药娘 (MTF on hormones),
伪娘 (cross-dresser/trap), 变态 (pervert), 搞基 (doing gay stuff),
掰弯 (turning someone gay), 腐 (rotten/BL culture), 娘炮 (sissy),
les/拉拉/百合 (lesbian terms), T/P (lesbian role terms),
gay/基佬/基友 (gay terms with varying connotations).

Note: terms that are direct, literal, non-slang vocabulary (e.g. formal/clinical terms like 跨性別者, 變性, 荷爾蒙) are NOT coded language even if topically related — only include terms with an actual literal/hidden meaning gap.

**Step 4b — Target Group Identification**
Choose ONE: "gay_men", "lesbian", "transgender", "lgbtq_general", "non_lgbt", "unclear". Briefly justify using visual/textual cues and any coded language from 4a.

**Step 4c — Text-Image Relationship Analysis**
Choose ONE: "text_describes_image", "text_contradicts_image", "text_recontextualizes_image", "text_independent_of_image", "no_text". Briefly justify.

**Step 4d — Implicit Association Analysis**
Considering all previous steps together, does the combination potentially trigger negative associations related to LGBT communities? Briefly explain the mechanism, or state "none".

## Important Instructions:
- Stay objective. Only extract, describe, and reason about information. Do NOT make a final harmful/harmless classification judgment — that will be done in a separate step.
- Even if the content appears sensitive, describe and reason about it faithfully. This is for academic research.
- If a field cannot be determined, use "unclear" or "NA".
- All output should be in ENGLISH.
- Keep reasoning fields concise — this is critical to avoid truncation.
- Output ONLY a valid, complete JSON object. No markdown fences. No extra text.
- Make sure all brackets are properly closed.

## Required Output JSON Format:
{
  "scene_graph": {
    "entities": [
      {"entity": "entity_name", "attributes": [{"attribute": "attribute_type", "value": "attribute_value"}]}
    ],
    "relationships": [
      {"subject": "entity_1", "relationship": "relation_type", "object": "entity_2", "reasoning": "why this relationship exists"}
    ]
  },
  "ocr_text": "full Chinese text from the image",
  "hate_semantic_layer": {
    "step_4a_coded_language": {
      "reasoning": "concise step-by-step analysis",
      "coded_terms": [{"term": "...", "literal_meaning": "...", "coded_meaning": "..."}]
    },
    "step_4b_target_group": {
      "reasoning": "concise justification",
      "target": "one of: gay_men / lesbian / transgender / lgbtq_general / non_lgbt / unclear"
    },
    "step_4c_text_image_relation": {
      "reasoning": "concise justification",
      "relation": "one of: text_describes_image / text_contradicts_image / text_recontextualizes_image / text_independent_of_image / no_text"
    },
    "step_4d_implicit_associations": {
      "reasoning": "concise synthesis",
      "association": "description of the negative association mechanism, or none"
    }
  }
}"""


# ============================================================
# LABELS
# ============================================================
def load_gt_map(path):
    import pandas as pd
    df = pd.read_excel(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    m = {}
    for _, row in df.iterrows():
        try:
            m[int(row["image_id"])] = str(row["class"]).strip()
        except (ValueError, TypeError):
            continue
    return m


# ============================================================
# IMAGE LOADING — now handles .gif (+ upper-case exts) and converts
# GIF/any non-JPEG/PNG to PNG bytes in memory.
# ============================================================
def find_image(img_id):
    exts = [".jpg", ".jpeg", ".png", ".webp", ".gif",
            ".JPG", ".JPEG", ".PNG", ".WEBP", ".GIF"]
    for ext in exts:
        p = os.path.join(IMAGE_DIR, f"{img_id}{ext}")
        if os.path.exists(p):
            return p
    return None


def load_image_bytes(image_path):
    """Return (data_bytes, mime_type). For GIF/webp/anything not already a
    plain JPEG or PNG, decode frame 0 and re-encode as PNG so the request
    path is uniform and multi-frame GIFs don't cause issues."""
    ext = os.path.splitext(image_path)[1].lower()
    if ext in (".jpg", ".jpeg"):
        with open(image_path, "rb") as f:
            return f.read(), "image/jpeg"
    if ext == ".png":
        with open(image_path, "rb") as f:
            return f.read(), "image/png"
    # gif / webp / other -> first frame -> PNG
    with Image.open(image_path) as im:
        im.seek(0)                      # first frame (no-op for static)
        rgb = im.convert("RGB")
        buf = io.BytesIO()
        rgb.save(buf, format="PNG")
        return buf.getvalue(), "image/png"


# ============================================================
# CALL
# ============================================================
def call_step1(client, image_path, prompt, max_tokens, thinking):
    image_data, mime_type = load_image_bytes(image_path)
    thinking_cfg = types.ThinkingConfig(thinking_budget=thinking)

    raw = ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=[
                    types.Part.from_bytes(data=image_data, mime_type=mime_type),
                    prompt,
                ],
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    max_output_tokens=max_tokens,
                    thinking_config=thinking_cfg,
                ),
            )
            raw = response.text.strip()
            cleaned = raw
            if cleaned.startswith("```json"):
                cleaned = cleaned[7:]
            elif cleaned.startswith("```"):
                cleaned = cleaned[3:]
            if cleaned.endswith("```"):
                cleaned = cleaned[:-3]
            cleaned = cleaned.strip()
            return json.loads(cleaned), True, raw
        except json.JSONDecodeError:
            if attempt < MAX_RETRIES:
                print(f"      JSON parse failed ({attempt}/{MAX_RETRIES}), "
                      f"len={len(raw)}, tail=...{raw[-80:]}")
                time.sleep(RETRY_DELAY)
            else:
                return None, False, raw
        except Exception as e:
            msg = str(e)
            if "503" in msg or "UNAVAILABLE" in msg or "429" in msg:
                wait = RETRY_DELAY * attempt
                print(f"      Server error ({attempt}), waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"      NON-RETRYABLE: {msg[:200]}")
                return None, False, msg
    return None, False, "Max retries exceeded"


# ============================================================
# MERGE
# ============================================================
def load_json(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def merge_and_save(path, new_records):
    data = load_json(path)
    new_ids = {r["image_id"] for r in new_records}
    # drop any prior entry (failed placeholder or otherwise) for these ids
    data = [r for r in data if r.get("image_id") not in new_ids]
    data.extend(new_records)
    data.sort(key=lambda x: x.get("image_id", 0))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    return data


# ============================================================
# BACKFILL ONE VERSION
# ============================================================
def backfill(client, version, ids, prompt, out_path, max_tokens, thinking, gt_map):
    print(f"\n{'#'*66}\n# BACKFILL {version.upper()} — ids {ids}\n{'#'*66}")
    new_records = []
    still_failed = []
    for img_id in ids:
        gt = gt_map.get(img_id, "UNKNOWN")
        image_path = find_image(img_id)
        if not image_path:
            print(f"  {version} img {img_id}: STILL not found (check filename)")
            still_failed.append(img_id)
            continue
        is_gif = image_path.lower().endswith(".gif")
        parsed, ok, raw = call_step1(client, image_path, prompt, max_tokens, thinking)
        if ok:
            parsed["image_id"] = img_id
            parsed["ground_truth"] = gt
            parsed["_step1_success"] = True
            if is_gif:
                parsed["_source_gif"] = True   # note for provenance
            new_records.append(parsed)
            print(f"  {version} img {img_id:>4} OK{'  (gif->png)' if is_gif else ''}")
        else:
            still_failed.append(img_id)
            print(f"  {version} img {img_id:>4} STILL FAILED — {raw[:100]}")
        time.sleep(4)

    if new_records:
        data = merge_and_save(out_path, new_records)
        ok_total = sum(1 for r in data if r.get("_step1_success"))
        print(f"\n  Merged {len(new_records)} into {os.path.basename(out_path)} "
              f"-> now {ok_total} successful records total.")
    if still_failed:
        print(f"  {version} STILL failing after backfill: {still_failed}")
    return still_failed


# ============================================================
# MAIN
# ============================================================
def main():
    client = genai.Client(api_key=API_KEY)
    gt_map = load_gt_map(LABELS_PATH)

    v2_left = backfill(client, "v2", V2_BACKFILL, PROMPT_V2, V2_OUT,
                       V2_MAX_TOKENS, 0, gt_map)
    v3_left = backfill(client, "v3", V3_BACKFILL, PROMPT_V3, V3_OUT,
                       V3_MAX_TOKENS, V3_THINKING, gt_map)

    print(f"\n{'='*66}\nBACKFILL COMPLETE")
    print(f"  V2 still-failed: {v2_left if v2_left else 'none'}")
    print(f"  V3 still-failed: {v3_left if v3_left else 'none'}")
    print("  If all clear, run the Step 2 full script (it will pick up the")
    print("  newly-merged records automatically).")
    print(f"{'='*66}")


if __name__ == "__main__":
    main()


##################################################################
# BACKFILL V2 — ids [13, 14, 129, 137, 179, 199, 221, 230, 224]
##################################################################
  v2 img   13 OK  (gif->png)
  v2 img   14 OK  (gif->png)
  v2 img  129 OK  (gif->png)
  v2 img  137 OK  (gif->png)
  v2 img  179 OK  (gif->png)
  v2 img  199 OK  (gif->png)
  v2 img  221 OK  (gif->png)
  v2 img  230 OK  (gif->png)
      JSON parse failed (1/4), len=22989, tail=...w_line_52",
        "attributes": [
          {
            "attribute": "type",
      JSON parse failed (2/4), len=22989, tail=...w_line_52",
        "attributes": [
          {
            "attribute": "type",
      JSON parse failed (3/4), len=22989, tail=...w_line_52",
        "attributes": [
          {
            "attribute": "type",
  v2 img  224 STILL FAILED — ```json
{
  "scene_graph": {
    "entities": [
      {
        "entity": "woman_1",
        "attribu

  Merged 8 into step1_v2_full.json -> now 238

Hm full step2 v2 v3

In [5]:
"""
HM FULL RUN — Step 2 (V2 and V3) -> the two HM table rows
=========================================================
Consumes step1_v2_full.json and step1_v3_full.json (from the full Step-1
script) and runs the Step 2 classifier on every successful Step-1 output,
for both versions. Produces, per version:
  - the 7-metric row for the HM results table
    (ACC, Macro-P/R/F1, Weighted-P/R/F1)
  - tie-break firing count + correctness
  - a misclassification list, auto-bucketed into:
      * "definitional_disagreement"  (hate GT -> predicted Non_Anti_LGBT
        where Step-1 already read the content as in-group / non-attacking)
      * "model_pragmatic_failure"    (everything else that's wrong)
    NOTE: the bucketing is a HEURISTIC first pass to speed up manual review;
    every wrong item still needs a human glance before it goes in the paper.

Handles both HSL shapes: V3 nested-with-reasoning, V2 flat. safe_get guards
against present-but-null fields (models sometimes emit "field": null).

Run in Google Colab with Google Drive mounted.
"""

import json
import os
import time
from collections import Counter

from google import genai
from google.genai import types
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)

# ============================================================
# CONFIG
# ============================================================


BASE_DIR = "/content/drive/MyDrive/MyThesis2026/Chinese/Test"
IMAGE_DIR = os.path.join(BASE_DIR, "Test_images")
MODEL_NAME = "gemini-2.5-flash"
MAX_RETRIES = 3
RETRY_DELAY = 15

INPUTS = {
    "v2": os.path.join(BASE_DIR, "step1_v2_full.json"),
    "v3": os.path.join(BASE_DIR, "step1_v3_full.json"),
}
OUTPUTS = {
    "v2": os.path.join(BASE_DIR, "step2_v2_full_results.json"),
    "v3": os.path.join(BASE_DIR, "step2_v3_full_results.json"),
}

LABELS = ["Homophobic", "Transphobic", "Non_Anti_LGBT"]
EDGE_TARGETS = {"lgbtq_general", "unclear"}


# ============================================================
# safe_get — treat present-but-null the same as missing
# ============================================================
def safe_get(d, key, default=None):
    if not isinstance(d, dict):
        return default
    v = d.get(key, default)
    return default if v is None else v


# ============================================================
# GUIDELINES + TEMPLATE (verbatim from the validated Step 2)
# ============================================================
CLASSIFICATION_GUIDELINES = """
## Classification Guidelines: Homophobic / Transphobic / Non_Anti_LGBT

You are classifying a Chinese internet meme into exactly ONE of three
classes, based on structured evidence extracted in a prior analysis step
(scene graph, OCR text, and a Hate Semantic Layer). Apply the following
definitions and decision procedure.

### Class Definitions

**Homophobic**: The content mocks, demeans, dehumanizes, sexually
objectifies without consent, expresses hostility or disgust toward, or
advocates exclusion/harm against gay men, lesbians, bisexual people, or
LGB people collectively, on the basis of sexual orientation or same-sex
attraction/relationships. This includes: trivializing or degrading gay
sexual roles or relationships, negative stereotyping (e.g., promiscuity,
predation, immorality), slurs, or "jokes" whose humor depends on treating
being gay/lesbian as shameful, disgusting, or inferior.

**Transphobic**: The content mocks, demeans, dehumanizes, denies the
legitimacy or existence of, expresses hostility or disgust toward, or
advocates exclusion/harm against transgender or gender-nonconforming
people, on the basis of gender identity or transition status. This
includes: mockery of gender transition or appearance, deliberate
mocking misgendering, framing trans identity as deception/mental
illness/predation, or hostile framing of trans participation in social
spaces (sports, bathrooms, etc.) that relies on disgust or dismissal
rather than neutral factual discussion.

**Non_Anti_LGBT**: Any of the following:
- Content unrelated to LGBT topics entirely.
- Content that references LGBT people/topics neutrally, factually, or
  informatively, without mockery or negative framing.
- Content that is affirming, celebratory, or in-group humor without
  derogatory intent (e.g., self-referential community slang used
  positively).
- Content that raises a genuinely neutral question or political/social
  discussion about LGBT-related policy WITHOUT relying on mockery,
  disgust framing, or dehumanization to make its point.
- Content where the evidence is genuinely insufficient to support a
  Homophobic or Transphobic judgment (default to this class when unsure
  — do not infer hate from ambiguous or absent evidence).

### Decision Procedure (apply in order)

1. **Check relevance**: If the scene graph, OCR text, and Hate Semantic
   Layer show no connection to LGBT people or topics at all -> Non_Anti_LGBT.
   Stop.

2. **Check valence**: Based on the coded language analysis, the
   text-image relationship, and the implicit associations described in
   Step 1, determine whether the content constructs a NEGATIVE
   (mocking / dehumanizing / hostile / degrading) association with the
   target group, or a NEUTRAL/POSITIVE one.
   - If neutral or positive -> Non_Anti_LGBT. Stop.
   - If negative, continue to step 3.

3. **Map target group to class** (only reached if valence is negative):
   - target = "gay_men" or "lesbian" -> Homophobic
   - target = "transgender" -> Transphobic
   - target = "lgbtq_general" or "unclear" but valence is clearly
     negative -> apply the TIE-BREAK rule below.
   - target = "non_lgbt" but valence somehow flagged negative toward an
     LGBT-adjacent theme -> re-examine; if truly no LGBT target,
     Non_Anti_LGBT.

### Tie-break rule (target = "lgbtq_general" or ambiguous, valence negative)

Ask: does the negative content specifically invoke gender-identity /
transition-specific themes (e.g., pronouns, transition, hormones,
gender-affirming care, "born as", bathrooms, single-sex sports
categories framed around gender identity)?
- If YES -> Transphobic.
- If NO, but it invokes sexual orientation / same-sex attraction or
  relationship themes -> Homophobic.
- If genuinely both or neither are clearly invoked -> Homophobic
  (default; document this case as ambiguous in your reasoning field so
  it can be reviewed).

### Important Instructions
- Base your judgment ONLY on the evidence provided (scene graph, OCR
  text, Hate Semantic Layer fields, and the image). Do not assume
  additional context not present in the evidence.
- Follow the decision procedure explicitly and show your reasoning for
  each step before giving the final label.
- Output ONLY a valid, complete JSON object. No markdown fences.
"""

STEP2_PROMPT_TEMPLATE = """You are an expert academic annotator classifying Chinese internet memes for a hate speech detection research study. You are given the image and a structured analysis (scene graph + Hate Semantic Layer) produced in a prior step. Apply the classification guidelines below with explicit Chain-of-Thought reasoning to produce the FINAL classification.

{guidelines}

## Structured Evidence from Step 1 (Scene Graph + Hate Semantic Layer)
```json
{step1_evidence}
```

## Required Output JSON Format
{{
  "step1_relevance_check": {{
    "reasoning": "Apply Decision Procedure Step 1 here.",
    "is_lgbt_relevant": true
  }},
  "step2_valence_check": {{
    "reasoning": "Apply Decision Procedure Step 2 here, using the coded language, text-image relation, and implicit associations evidence.",
    "valence": "negative | neutral_or_positive"
  }},
  "step3_target_mapping": {{
    "reasoning": "Apply Decision Procedure Step 3 (and the tie-break rule if applicable) here.",
    "tie_break_applied": false
  }},
  "final_label": "Homophobic | Transphobic | Non_Anti_LGBT",
  "confidence": "high | medium | low"
}}

Output ONLY the JSON object above, fully filled in. No markdown fences, no extra text."""


# ============================================================
# HSL NORMALIZATION (version-aware)
# ============================================================
def normalize_hsl(step1_output, version):
    hsl = safe_get(step1_output, "hate_semantic_layer", {})
    if version == "v3":
        coded = safe_get(hsl, "step_4a_coded_language", {})
        target = safe_get(hsl, "step_4b_target_group", {})
        relation = safe_get(hsl, "step_4c_text_image_relation", {})
        assoc = safe_get(hsl, "step_4d_implicit_associations", {})
        return {
            "coded_language_terms": safe_get(coded, "coded_terms", []),
            "coded_language_reasoning": safe_get(coded, "reasoning", ""),
            "target_group": safe_get(target, "target", "unclear"),
            "target_group_reasoning": safe_get(target, "reasoning", ""),
            "text_image_relation": safe_get(relation, "relation", "unclear"),
            "text_image_relation_reasoning": safe_get(relation, "reasoning", ""),
            "implicit_associations": safe_get(assoc, "association", "none"),
            "implicit_associations_reasoning": safe_get(assoc, "reasoning", ""),
        }
    else:  # v2 flat
        return {
            "coded_language_terms": safe_get(hsl, "coded_language", []),
            "coded_language_reasoning": "N/A (V2 has no explicit reasoning field)",
            "target_group": safe_get(hsl, "target_group", "unclear"),
            "target_group_reasoning": "N/A (V2 has no explicit reasoning field)",
            "text_image_relation": safe_get(hsl, "text_image_relation", "unclear"),
            "text_image_relation_reasoning": "N/A (V2 has no explicit reasoning field)",
            "implicit_associations": safe_get(hsl, "implicit_associations", "none"),
            "implicit_associations_reasoning": "N/A (V2 has no explicit reasoning field)",
        }


def get_target(step1_output, version):
    return normalize_hsl(step1_output, version)["target_group"]


def build_evidence(step1_output, version):
    return {
        "scene_graph": safe_get(step1_output, "scene_graph", {}),
        "ocr_text": safe_get(step1_output, "ocr_text", ""),
        "hate_semantic_layer": normalize_hsl(step1_output, version),
    }


# ============================================================
# GEMINI
# ============================================================
def find_image(img_id):
    for ext in [".jpg", ".jpeg", ".png", ".webp"]:
        p = os.path.join(IMAGE_DIR, f"{img_id}{ext}")
        if os.path.exists(p):
            return p
    return None


def call_step2(client, image_path, step1_output, version):
    evidence = build_evidence(step1_output, version)
    prompt = STEP2_PROMPT_TEMPLATE.format(
        guidelines=CLASSIFICATION_GUIDELINES,
        step1_evidence=json.dumps(evidence, indent=2, ensure_ascii=False),
    )
    with open(image_path, "rb") as f:
        image_data = f.read()
    ext = os.path.splitext(image_path)[1].lower()
    mime_type = "image/png" if ext == ".png" else "image/jpeg"

    raw = ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=[
                    types.Part.from_bytes(data=image_data, mime_type=mime_type),
                    prompt,
                ],
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    max_output_tokens=2048,
                    thinking_config=types.ThinkingConfig(thinking_budget=1024),
                ),
            )
            raw = response.text.strip()
            cleaned = raw
            if cleaned.startswith("```json"):
                cleaned = cleaned[7:]
            elif cleaned.startswith("```"):
                cleaned = cleaned[3:]
            if cleaned.endswith("```"):
                cleaned = cleaned[:-3]
            cleaned = cleaned.strip()
            return json.loads(cleaned), True, raw
        except json.JSONDecodeError:
            if attempt < MAX_RETRIES:
                print(f"      JSON parse failed ({attempt}/{MAX_RETRIES})...")
                time.sleep(RETRY_DELAY)
            else:
                return None, False, raw
        except Exception as e:
            msg = str(e)
            if "503" in msg or "UNAVAILABLE" in msg or "429" in msg:
                wait = RETRY_DELAY * attempt
                print(f"      Server error ({attempt}), waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"      NON-RETRYABLE: {msg[:200]}")
                return None, False, msg
    return None, False, "Max retries exceeded"


# ============================================================
# CHECKPOINT
# ============================================================
def load_ckpt(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        done = {r["image_id"] for r in data if r.get("_step2_success")}
        return data, done
    return [], set()


def save(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


# ============================================================
# HEURISTIC BUCKETING for misclassifications
# ============================================================
def bucket_wrong(rec, version):
    """First-pass heuristic. Definitional-disagreement candidates: a hate GT
    predicted as Non_Anti_LGBT where Step-1's implicit_associations read as
    'none' (i.e. Step-1 itself judged the content non-attacking). Everything
    else wrong -> model_pragmatic_failure. HUMAN REVIEW STILL REQUIRED."""
    gt = rec["ground_truth"]
    pred = rec["final_label"]
    if gt in ("Homophobic", "Transphobic") and pred == "Non_Anti_LGBT":
        assoc = str(rec.get("step1_implicit_assoc", "")).strip().lower()
        if assoc in ("none", "", "na"):
            return "definitional_disagreement"
        return "model_pragmatic_failure"
    return "model_pragmatic_failure"


# ============================================================
# RUN ONE VERSION
# ============================================================
def run_version(client, version):
    in_path = INPUTS[version]
    out_path = OUTPUTS[version]

    with open(in_path, "r", encoding="utf-8") as f:
        stage1 = json.load(f)
    ok1 = [r for r in stage1 if r.get("_step1_success")]
    bad1 = [r for r in stage1 if not r.get("_step1_success")]

    print(f"\n{'#'*70}\n# STEP 2 — {version.upper()}\n{'#'*70}")
    print(f"Step-1 usable: {len(ok1)}, Step-1 failures (excluded): {len(bad1)} "
          f"{sorted(r['image_id'] for r in bad1) if bad1 else ''}")

    results, done = load_ckpt(out_path)
    if done:
        print(f"Resuming {version} Step 2: {len(done)} done, skipping.\n")

    for s1 in sorted(ok1, key=lambda x: x["image_id"]):
        img_id, gt = s1["image_id"], s1["ground_truth"]
        if img_id in done:
            continue
        image_path = find_image(img_id)
        if not image_path:
            print(f"[SKIP] {version} image {img_id}: not found")
            continue

        s1_target = get_target(s1, version)
        s1_assoc = normalize_hsl(s1, version)["implicit_associations"]
        parsed, ok, raw = call_step2(client, image_path, s1, version)

        rec = {
            "image_id": img_id,
            "ground_truth": gt,
            "step1_target": s1_target,
            "step1_target_is_edge": s1_target in EDGE_TARGETS,
            "step1_implicit_assoc": s1_assoc,
        }
        if ok:
            rec["final_label"] = safe_get(parsed, "final_label", "PARSE_ERROR")
            rec["tie_break_applied"] = bool(
                safe_get(safe_get(parsed, "step3_target_mapping", {}),
                         "tie_break_applied", False))
            rec["confidence"] = safe_get(parsed, "confidence", "N/A")
            rec["step2_raw"] = parsed
            rec["_step2_success"] = True
            mark = "OK" if rec["final_label"] == gt else f"WRONG(GT={gt})"
            tb = "TB" if rec["tie_break_applied"] else "  "
            print(f"  {version} img {img_id:>4} {tb} -> {rec['final_label']:<14} {mark}")
        else:
            rec["final_label"] = "ERROR"
            rec["tie_break_applied"] = False
            rec["_step2_success"] = False
            rec["_error"] = raw[:300]
            print(f"  {version} img {img_id:>4} FAILED")

        results.append(rec)
        save(out_path, results)
        time.sleep(4)

    return results


# ============================================================
# REPORT ONE VERSION
# ============================================================
def report_version(results, version):
    ok = [r for r in results if r.get("_step2_success")]
    valid = [r for r in ok if r["final_label"] in LABELS]

    print(f"\n{'='*70}\nRESULTS — {version.upper()}\n{'='*70}")
    print(f"n classified = {len(valid)}  "
          f"(excluded {len(ok)-len(valid)} non-label, {len(results)-len(ok)} errors)")

    if not valid:
        print("No valid rows — cannot compute metrics.")
        return

    y_true = [r["ground_truth"] for r in valid]
    y_pred = [r["final_label"] for r in valid]

    acc = accuracy_score(y_true, y_pred)
    mp, mr, mf1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, average="macro", zero_division=0)
    wp, wr, wf1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, average="weighted", zero_division=0)

    print(f"\n  === HM TABLE ROW ({version.upper()}) ===")
    print(f"  ACC={acc:.4f}  MP={mp:.4f}  MR={mr:.4f}  MF1={mf1:.4f}  "
          f"WP={wp:.4f}  WR={wr:.4f}  WF1={wf1:.4f}")
    print("\n  Per-class:")
    print(classification_report(y_true, y_pred, labels=LABELS, zero_division=0))

    print("  Confusion (GT rows x pred cols):")
    cm = Counter((r["ground_truth"], r["final_label"]) for r in valid)
    print("    GT \\ pred     " + "".join(f"{l[:11]:>13}" for l in LABELS))
    for gt in LABELS:
        row = "".join(f"{cm.get((gt, pr), 0):>13}" for pr in LABELS)
        print(f"    {gt:<14}{row}")

    # Tie-break
    fired = [r for r in ok if r.get("tie_break_applied")]
    print(f"\n  Tie-break fired: {len(fired)}")
    if fired:
        n_ok = sum(1 for r in fired if r["final_label"] == r["ground_truth"])
        print(f"  Tie-break correct: {n_ok}/{len(fired)}")
        for r in fired:
            hit = "OK" if r["final_label"] == r["ground_truth"] else "WRONG"
            print(f"    img {r['image_id']:>4} GT={r['ground_truth']:<14} "
                  f"-> {r['final_label']:<14} {hit}")

    # Misclassification buckets (heuristic)
    wrong = [r for r in valid if r["final_label"] != r["ground_truth"]]
    print(f"\n  Misclassifications: {len(wrong)} (heuristic buckets — REVIEW MANUALLY)")
    buckets = {}
    for r in wrong:
        b = bucket_wrong(r, version)
        buckets.setdefault(b, []).append(r)
    for b, rows in buckets.items():
        print(f"    [{b}] : {len(rows)}")
        for r in sorted(rows, key=lambda x: x["image_id"]):
            print(f"        img {r['image_id']:>4} GT={r['ground_truth']:<14} "
                  f"-> {r['final_label']:<14} | step1_target={r['step1_target']:<14} "
                  f"| assoc='{str(r.get('step1_implicit_assoc',''))[:40]}'")

    print(f"\n  Metrics dict (copy-paste): "
          f"{{'ACC':{acc:.4f},'MP':{mp:.4f},'MR':{mr:.4f},'MF1':{mf1:.4f},"
          f"'WP':{wp:.4f},'WR':{wr:.4f},'WF1':{wf1:.4f}}}")


# ============================================================
# MAIN
# ============================================================
def main():
    client = genai.Client(api_key=API_KEY)
    all_results = {}
    for version in ("v2", "v3"):
        if not os.path.exists(INPUTS[version]):
            print(f"[SKIP] {version}: {INPUTS[version]} not found — "
                  f"run the Step 1 full script first.")
            continue
        all_results[version] = run_version(client, version)

    for version in ("v2", "v3"):
        if version in all_results:
            report_version(all_results[version], version)

    print(f"\n{'='*70}")
    print("DONE. Fill the HM table rows from the 'HM TABLE ROW' lines above:")
    print("  Scene Graph without CoT  <- V2")
    print("  Scene Graph with CoT     <- V3")
    print(f"{'='*70}")


if __name__ == "__main__":
    main()


######################################################################
# STEP 2 — V2
######################################################################
Step-1 usable: 238, Step-1 failures (excluded): 1 [224]
      Server error (1), waiting 15s...
      Server error (2), waiting 30s...
  v2 img    1    -> Homophobic     OK
  v2 img    2    -> Non_Anti_LGBT  WRONG(GT=Homophobic)
      Server error (1), waiting 15s...
  v2 img    3    -> Non_Anti_LGBT  WRONG(GT=Homophobic)
  v2 img    4    -> Homophobic     OK
  v2 img    5    -> Transphobic    OK
  v2 img    6    -> Transphobic    OK
  v2 img    7    -> Non_Anti_LGBT  OK
  v2 img    8    -> Non_Anti_LGBT  WRONG(GT=Homophobic)
  v2 img    9    -> Transphobic    OK
  v2 img   10    -> Non_Anti_LGBT  WRONG(GT=Homophobic)
  v2 img   11    -> Non_Anti_LGBT  WRONG(GT=Homophobic)
  v2 img   12    -> Non_Anti_LGBT  OK
[SKIP] v2 image 13: not found
[SKIP] v2 image 14: not found
  v2 img   15    -> Homophobic     OK
      Server error (1), wa